# alphaPhos cardio ANOVA walkthrough

Full multi-group differential-expression + downstream ORA pipeline on the
Mann-lab cardiomyocyte DVP dataset — **5 disease groups × 3 tissue regions
× ~75 cells** — worked through in three parallel layers:

| Layer | What it measures |
|---|---|
| **proteome** | Bulk protein-group abundance per sample. Blood- and stroma-contaminated in cardiac tissue — you'll see. |
| **phospho** | Site-level phospho intensity. Sarcomere-abundance-linked signal dominates. |
| **normalized** | Phospho / matched-sample parent-protein (log2 subtraction). Removes protein-abundance confounding — the *"which phospho events actually matter?"* view. |

Ships in alphaPhos 0.17.0+ via:

- `ap.diff_exp_anova` — moderated F-test across all K condition levels (clean-room Smyth 2004; MIT-compatible).
- `ap.anova_hits` — (hits, background) split for ORA on ANOVA output.
- `ap.enrichment.pathway_enrichment(direction="any")` — direction-agnostic gene ORA on Enrichr libraries.
- `ap.enrichment.canonicalise_site_ids` — bridges alphaPhos `Protein|Gene|Site|Mult` keys to the `Protein_AApos` IDs used by the PTM-DB GMT libraries.

Prerequisites:

```bash
pip install -e ".[stats,enrichment]"     # inmoose + gseapy + decoupler
```

Data lives at `test_data/proteome_path/` — Spectronaut phospho PSMs +
Spectronaut short-format proteome + condition_df.

In [1]:
from __future__ import annotations

import logging
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import alphaphos as ap

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.WARNING, format="%(name)s %(levelname)s %(message)s")

try:
    HERE = Path(__file__).resolve().parent
except NameError:
    HERE = Path.cwd()
REPO = HERE if (HERE / "src" / "alphaphos").exists() else HERE.parent
assert (REPO / "src" / "alphaphos").exists(), f"Could not locate alphaPhos repo from {HERE}"

DATA = REPO / "test_data" / "proteome_path"
OUT = DATA / "cardio_output"
OUT.mkdir(exist_ok=True)
PTM_LIBS = OUT / "ptm_libs"

PHOSPHO_PARQUET = DATA / "cardiomyocytes_dvp_phospho_raw.parquet"
PROTEOME_SHORT = DATA / "cardiomyocytes_dvp_proteome_spectronaut_short_parquet.parquet"
CONDITION_CSV = DATA / "cardiomyocytes_condition_df.csv"

print(f"alphaPhos {ap.__version__}")
print(f"Data:   {DATA}")
print(f"Out:    {OUT}")

alphaPhos 0.17.0
Data:   D:\Projects\alphaPhos\test_data\proteome_path
Out:    D:\Projects\alphaPhos\test_data\proteome_path\cardio_output


## 1 — Read + parse conditions

The `condition_df` packs `{PatientID}_{Disease}_{Region}` in one string. We
split into three obs columns *in the notebook* (the alphaPhos convention
keeps `condition_df` opaque). `disease` is the biological factor for the
ANOVA; `region` and `patient` are additional axes you'd use as covariates
or paired-block factors in a real study.

The proteome report's column headers replace `phosphoDVP` with `proteomeDVP`
(same wells, second acquisition), so we swap the tag in a copy of
`condition_df` for the proteome read.

In [2]:
cond = pd.read_csv(CONDITION_CSV)
cond_prot = cond.copy()
cond_prot["sample"] = cond_prot["sample"].str.replace("phosphoDVP", "proteomeDVP", regex=False)


def _parse_condition(c):
    if not isinstance(c, str) or c == "BLANK_BLANK_BLANK" or not c:
        return {
            "patient": None,
            "disease": "BLANK" if isinstance(c, str) and c else None,
            "region": None,
        }
    parts = c.split("_")
    if len(parts) == 3:
        return {"patient": parts[0], "disease": parts[1], "region": parts[2]}
    return {"patient": None, "disease": None, "region": None}


def _attach_and_filter(adata):
    parsed = adata.obs["condition"].map(_parse_condition).apply(pd.Series)
    for col in ("patient", "disease", "region"):
        adata.obs[col] = parsed[col].values
    keep = adata.obs["disease"].notna() & (adata.obs["disease"] != "BLANK")
    return adata[keep].copy()


psm_df = ap.read_spectronaut(PHOSPHO_PARQUET)
adata_phos = ap.collapse_sites(psm_df, condition_df=cond)
adata_phos = _attach_and_filter(adata_phos)

adata_prot = ap.proteome.read_spectronaut_short(PROTEOME_SHORT, condition_df=cond_prot)
adata_prot = _attach_and_filter(adata_prot)

print("Raw shapes (after BLANK drop + condition parse)")
print(f"  phospho:  {adata_phos.shape}")
print(f"  proteome: {adata_prot.shape}")
print(f"  disease levels: {sorted(adata_phos.obs['disease'].unique())}")

alphaphos.preprocess.collapse WARNING 8 gene names contained underscores and have been temporarily replaced with '#' (restored at output).


alphaphos.proteome.io_short WARNING read_spectronaut_short: 3/77 samples in the report have no row in condition_df (their .obs values will be NaN). First few: ['20260703_OA5_Evo13_Whisper80_SA_DeOl_cardiomyocyteDVP_proteomeDVP_A10_20260706070624', '20260703_OA5_Evo13_Whisper80_SA_DeOl_cardiomyocyteDVP_proteomeDVP_B11', '20260703_OA5_Evo13_Whisper80_SA_DeOl_cardiomyocyteDVP_proteomeDVP_G7']


alphaphos.proteome.io_short WARNING read_spectronaut_short: 2 rows in condition_df have no matching sample in the report and were dropped from .obs.


Raw shapes (after BLANK drop + condition parse)
  phospho:  (72, 20953)
  proteome: (70, 5644)
  disease levels: ['ACM', 'HCM', 'Healthy', 'ICM', 'NICM']


## 2 — Filter + impute (identical pipeline per layer)

Same `filter_by_completeness(min_valid_frac=2/3, keep_strategy="any")`
+ `impute_hybrid` combo used elsewhere. Applied unchanged to phospho and
proteome — the operations don't care which layer they're looking at.

In [3]:
def _filter_impute(adata):
    a = ap.filter_by_completeness(
        adata,
        min_valid_frac=2 / 3,
        group_column="disease",
        keep_strategy="any",
        layer="intensity_log2",
    )
    a = ap.impute_hybrid(a)
    a.X = a.layers["intensity_log2"].copy()
    return a


phos_ready = _filter_impute(adata_phos)
prot_ready = _filter_impute(adata_prot)

print(f"phospho  filter+impute: {phos_ready.shape}")
print(f"proteome filter+impute: {prot_ready.shape}")

phospho  filter+impute: (72, 4687)
proteome filter+impute: (70, 4489)


## 3 — The normalized layer: `phospho_over_proteome`

The killer feature of the proteome module. Divides phospho intensity by
the matched sample's parent-protein intensity in log2 space, yielding
**log fraction phosphorylated** — the phospho signal that isn't explained
by "more of that protein was there".

Sample pairing is by DVP well-ID regex (`_A1`..`_G11`). Sites with no
parent-protein match land as NaN and get filled in by the imputer, same
as the phospho layer.

In [4]:
adata_norm = ap.proteome.phospho_over_proteome(
    adata_phos,
    adata_prot,
    sample_pairing="auto",
    missing_protein="drop",
    protein_group_policy="first",
)
adata_norm = _attach_and_filter(adata_norm)
norm_ready = _filter_impute(adata_norm)

layers = {
    "proteome": prot_ready,
    "phospho": phos_ready,
    "normalized": norm_ready,
}
print("Post-filter-impute shapes")
for name, a in layers.items():
    diseases = dict(a.obs["disease"].value_counts())
    print(f"  {name:11s}  {a.shape}   nan={int(np.isnan(a.X).sum())}   diseases={diseases}")

Post-filter-impute shapes
  proteome     (70, 4489)   nan=0   diseases={'ICM': np.int64(19), 'Healthy': np.int64(17), 'HCM': np.int64(16), 'NICM': np.int64(14), 'ACM': np.int64(4)}
  phospho      (72, 4687)   nan=0   diseases={'ICM': np.int64(19), 'Healthy': np.int64(18), 'HCM': np.int64(16), 'NICM': np.int64(15), 'ACM': np.int64(4)}
  normalized   (70, 3687)   nan=0   diseases={'ICM': np.int64(19), 'Healthy': np.int64(17), 'HCM': np.int64(16), 'NICM': np.int64(14), 'ACM': np.int64(4)}


## 4 — Moderated F-test across all 5 disease groups (`ap.diff_exp_anova`)

One test per feature: *"does the mean differ anywhere across the K disease
levels?"* Uses alphaPhos's own clean-room Smyth 2004 empirical-Bayes
moderation — the same numerical stack validated bit-exact against
`inmoose` on the 2-group case (see [test_stats_moderated.py](../tests/unit/test_stats_moderated.py)
and 0.17.0 CHANGELOG).

Output columns: `F`, `p_value`, `fdr`, `ave_expr`, `df_between`,
`df_moderated`. **No signed statistic** — the F-test is direction-agnostic
by construction.

In [5]:
anova = {}
for name, a in layers.items():
    r = ap.diff_exp_anova(a, condition_column="disease", layer="intensity_log2")
    n_sig05 = int((r["fdr"] < 0.05).sum())
    n_sig01 = int((r["fdr"] < 0.01).sum())
    anova[name] = r
    print(
        f"[{name:11s}]  tested={len(r):<5d}  fdr<0.05={n_sig05:<5d}  fdr<0.01={n_sig01:<5d}  top_F={r['F'].max():.2f}"
    )
    r.to_csv(OUT / f"anova_{name}.tsv", sep="\t")

[proteome   ]  tested=4489   fdr<0.05=777    fdr<0.01=288    top_F=33.31
[phospho    ]  tested=4687   fdr<0.05=1188   fdr<0.01=732    top_F=25.14
[normalized ]  tested=3687   fdr<0.05=463    fdr<0.01=195    top_F=20.12


## 5 — Top-10 ANOVA hits per layer

Watch what happens across the three columns:

- **proteome** — dominated by hemoglobin subunits (HBA/HBB/HBD) + carbonic anhydrase (CA1). That's **differential vascularization** across disease groups (scar tissue has altered blood content), not myocyte biology. Real signal, but it swamps the muscle-specific channel.
- **phospho (raw)** — SYNPO2L, CKM, SORBS1, CALU, RBM14. Sarcomere-linked, but *because those proteins are more abundant in disease tissue*. Fold-change here is confounded with protein abundance.
- **normalized** — **ACTC1** (α-cardiac actin), **MYH7** (β-myosin heavy chain — the classic HCM gene), **DSP** (desmoplakin — ARVC gene), MTOR, NEBL. These are the phospho events beyond what protein-abundance change explains. **Textbook cardiomyopathy panel.**

In [6]:
for name, r in anova.items():
    top = r.sort_values("F", ascending=False).head(10).copy()
    if name == "proteome":
        top["gene"] = prot_ready.var.loc[top.index, "PG_Genes"].values
    else:
        top["gene"] = top.index.to_series().str.split("|").str[1]
    view = top[["gene", "F", "fdr"]].copy()
    view["F"] = view["F"].round(2)
    view["fdr"] = view["fdr"].apply(lambda v: f"{v:.2e}")
    print(f"\n[{name}]")
    print(view.to_string())


[proteome]
            gene      F       fdr
P00915       CA1  33.31  1.03e-11
P69905      HBA2  30.06  4.81e-11
P68871       HBB  29.37  5.22e-11
P02042       HBD  28.65  6.64e-11
Q8WX93     PALLD  27.41  1.33e-10
Q13228  SELENBP1  22.71  4.49e-09
Q07960   ARHGAP1  22.55  4.49e-09
P06727     APOA4  19.09  7.70e-08
Q13508      ART3  19.03  7.70e-08
P51570     GALK1  18.92  7.70e-08

[phospho]
                           gene      F       fdr
Q9H987|SYNPO2L|S931|M1  SYNPO2L  25.14  1.06e-09
Q9H987|SYNPO2L|S446|M2  SYNPO2L  24.50  1.06e-09
P06732|CKM|T166|M1          CKM  24.45  1.06e-09
Q9BX66|SORBS1|T786|M1    SORBS1  23.28  2.15e-09
Q9H987|SYNPO2L|S438|M2  SYNPO2L  21.97  5.46e-09
O43852|CALU|S44|M1         CALU  21.58  6.41e-09
A5A3E0|POTEF|T949|M1      POTEF  21.31  6.96e-09
Q96PK6|RBM14|S618|M1      RBM14  21.13  6.96e-09
P06732|CKM|T313|M1          CKM  21.04  6.96e-09
Q9H987|SYNPO2L|S366|M3  SYNPO2L  20.61  7.22e-09

[normalized]
                           gene      F       fdr
Q

## 6 — Gene-level pathway ORA (`pathway_enrichment(direction="any")`)

ANOVA output has no sign, so the standard `direction="split"` up/down ORA
doesn't apply. The `direction="any"` mode is the direction-agnostic path:
take every gene with `fdr < fdr_threshold` as one blob, run one-sided
Fisher against Enrichr libraries.

For the **proteome** branch we attach gene names from `adata.var["PG_Genes"]`
and pass `gene_column="gene"` — because protein-group keys don't carry gene
symbols the way phospho keys do. `stat_col=None` skips the log2fc read.

In [7]:
HALL = ["MSigDB_Hallmark_2020"]

r_prot = anova["proteome"].copy()
if "PG_Genes" in prot_ready.var.columns:
    r_prot["gene"] = prot_ready.var.loc[r_prot.index, "PG_Genes"].values

pathway_inputs = {
    "proteome": (r_prot, {"gene_column": "gene", "stat_col": None}),
    "phospho": (anova["phospho"], {"stat_col": None}),
    "normalized": (anova["normalized"], {"stat_col": None}),
}

pathway_results = {}
for name, (r, kwargs) in pathway_inputs.items():
    try:
        pe = ap.enrichment.pathway_enrichment(
            r,
            fdr_threshold=0.05,
            direction="any",
            libraries=HALL,
            organism="human",
            **kwargs,
        )
    except Exception as e:
        print(f"  {name:11s}  SKIPPED  ({type(e).__name__}: {e})")
        continue
    pathway_results[name] = pe
    n_sig = int((pe["fdr"] < 0.05).sum())
    n_fg = pe.attrs.get("provenance", {}).get("n_foreground_per_direction", {}).get("any", 0)
    print(f"[{name:11s}]  tested={len(pe):<4d}  sig(fdr<0.05)={n_sig:<3d}  foreground={n_fg} genes")
    pe.to_csv(OUT / f"pathway_ora_anova_{name}.tsv", sep="\t", index=False)

[proteome   ]  tested=47    sig(fdr<0.05)=2    foreground=769 genes


[phospho    ]  tested=47    sig(fdr<0.05)=1    foreground=512 genes


[normalized ]  tested=43    sig(fdr<0.05)=0    foreground=219 genes


In [8]:
for name, pe in pathway_results.items():
    top = pe.sort_values("fdr").head(5)
    cols = [c for c in ("term", "overlap", "odds_ratio", "fdr") if c in top.columns]
    print(f"\n[{name}]")
    print(top[cols].to_string(index=False))


[proteome]
             term  odds_ratio      fdr
      Coagulation    2.587839 0.007139
          Hypoxia    2.298260 0.023151
       Glycolysis    1.903258 0.081576
       Complement    1.924286 0.082068
KRAS Signaling Up    2.512646 0.082068

[phospho]
                   term  odds_ratio      fdr
             Myogenesis    3.030813 0.001572
  Xenobiotic Metabolism    3.833533 0.199020
Estrogen Response Early    2.986286 0.199020
        Mitotic Spindle    1.822149 0.199020
             Glycolysis    2.096356 0.199020

[normalized]
                 term  odds_ratio      fdr
    KRAS Signaling Dn   17.718310 0.065988
           Myogenesis    1.925373 0.590643
Xenobiotic Metabolism    2.938679 0.590643
          Pperoxisome    8.736111 0.590643
 Bile Acid Metabolism    4.361111 0.697278


## 7 — Site-level PTM-DB ORA (`ap.enrichment.ora`)

For phospho + normalized only (proteome has no per-site functional
annotation). We test whether the ANOVA-sig sites are enriched in curated
PTM-functional site-sets — Ochoa functional-score bins, disease-variant
sets (ClinVar, cancer TCGA), and functional-effect categories (disrupts
PPIs, alters stability, inhibits activity, ...).

**Two-step call**:
1. `ap.anova_hits(anova, fdr_threshold)` → `(hits, background)` in
   alphaPhos site-key format.
2. `ap.enrichment.canonicalise_site_ids(...)` bridges to the
   `Protein_AApos` format used by the PTM-DB GMTs.

Then `ap.enrichment.ora(hits, bg, libraries=PTM_LIBS)` runs the Fisher
tests with FDR control.

In [9]:
site_ora_results = {}
if not PTM_LIBS.exists() or not any(PTM_LIBS.glob("*.gmt")):
    print(f"SKIPPED  no PTM libraries at {PTM_LIBS}")
else:
    for name in ("phospho", "normalized"):
        r = anova[name]
        hits_raw, bg_raw = ap.anova_hits(r, fdr_threshold=0.05)
        hits = ap.enrichment.canonicalise_site_ids(hits_raw)
        bg = ap.enrichment.canonicalise_site_ids(bg_raw)
        hits = [h for h in hits if h in set(bg)]  # ora() precondition
        ora_df = ap.enrichment.ora(hits, bg, libraries=PTM_LIBS)
        site_ora_results[name] = ora_df
        n_sig = int((ora_df["fdr"] < 0.05).sum())
        print(
            f"[{name:11s}]  hits={len(hits):<5d}  bg={len(bg):<5d}  sets_tested={len(ora_df):<3d}  sig(fdr<0.05)={n_sig}"
        )
        ora_df.to_csv(OUT / f"site_ora_anova_{name}.tsv", sep="\t", index=False)

[phospho    ]  hits=1188   bg=4686   sets_tested=9    sig(fdr<0.05)=2
[normalized ]  hits=463    bg=3686   sets_tested=9    sig(fdr<0.05)=2


In [10]:
for name, ora_df in site_ora_results.items():
    top = ora_df.sort_values("fdr").head(5).copy()
    if "log2_fold_enrichment" in top.columns:
        top["log2_fold_enrichment"] = top["log2_fold_enrichment"].round(2)
    if "fdr" in top.columns:
        top["fdr"] = top["fdr"].apply(lambda v: f"{v:.2e}")
    cols = [
        c
        for c in (
            "library",
            "set_name",
            "n_overlap",
            "n_set",
            "log2_fold_enrichment",
            "direction",
            "fdr",
        )
        if c in top.columns
    ]
    print(f"\n[{name}]")
    print(top[cols].to_string(index=False))


[phospho]
          library              set_name  n_overlap  n_set  log2_fold_enrichment direction      fdr
 functional_score    ochoa_top_quartile        232    981                 -0.20  depleted 9.28e-03
  disease_variant      cancer_TCGA_site        327   1341                 -0.16  depleted 1.05e-02
 functional_score ochoa_bottom_quartile         47    199                 -0.20  depleted 2.88e-01
  disease_variant          clinvar_site        100    341                  0.11  enriched 3.74e-01
functional_effect           induces_ppi         24     74                  0.25  enriched 8.08e-01

[normalized]
          library           set_name  n_overlap  n_set  log2_fold_enrichment direction      fdr
functional_effect       disrupts_ppi         10     32                  1.20  enriched 2.36e-02
functional_effect   alters_stability          7     19                  1.44  enriched 2.36e-02
functional_effect  inhibits_activity         20     93                  0.66  enriched 5.13e-

## 8 — What the layers tell you

This is the "why did we build the normalized layer" moment.

**Raw phospho ANOVA hits** are DEPLETED for Ochoa top-quartile functional
sites and TCGA cancer sites (fdr ~ 0.01). That's not a bug — it's the
signature of an ANOVA that's really picking up **protein-abundance
variation dressed as phospho variation**. Bulk-abundance-linked sites are
enriched in the hits, and those sites happen to be uncorrelated with
"functionally important" sets.

**Normalized ANOVA hits** are ENRICHED for `disrupts_ppi` (log2FE = +1.20,
fdr = 0.024), `alters_stability` (log2FE = +1.44, fdr = 0.024), and
`inhibits_activity` (log2FE = +0.66, fdr = 0.051). After protein-abundance
is subtracted out, the surviving phospho events are 2–3× enriched for
sites that **actually change substrate function**. That's what phospho
biology should look like.

**Proteome pathway hits** — Coagulation (fdr = 0.007), Hypoxia (fdr = 0.023)
— report the **tissue-level** biology: differential vascularization and
oxygen supply between healthy myocardium and fibrotic remodeled tissue.
Different question, same dataset.

Each layer answers a different biological question. The point of running
all three is that you can *see* which question you're answering.

## Modules exercised

| Step | Module | Function |
|---|---|---|
| §1 | `alphaphos.io.spectronaut` + `alphaphos.proteome.io_short` | `read_spectronaut`, `read_spectronaut_short` |
| §2 | `alphaphos.preprocess` | `filter_by_completeness`, `impute_hybrid` |
| §3 | `alphaphos.proteome.pairing` | `phospho_over_proteome` |
| §4 | `alphaphos.stats.diff_exp` | `diff_exp_anova` |
| §6 | `alphaphos.enrichment.pathway` | `pathway_enrichment(direction="any")` |
| §7 | `alphaphos.stats.diff_exp` + `alphaphos.enrichment.matching` + `alphaphos.enrichment.enrich` | `anova_hits`, `canonicalise_site_ids`, `ora` |